# 🏥 Organ Classifier V2 — FAST Local PC Training

## Why fast:
- **Cell 5 (run ONCE ~15-30 min):** Reads all images → resizes → saves to `organ_cache.h5`
- **Cell 6+ (every run):** Loads cache into RAM → zero disk I/O during training
- Expected epoch time: **2–5 min** on GPU

## Install deps first:
```
pip install torch torchvision medmnist h5py scikit-learn matplotlib seaborn pillow tqdm psutil
```

## Only edit Cell 2 — set your 3 paths

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import torch, sys, psutil
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB')
else:
    print('NO GPU — go to Runtime > Change runtime type > T4 GPU')
ram = psutil.virtual_memory()
print(f'RAM     : {ram.total/1e9:.1f} GB total  |  {ram.available/1e9:.1f} GB free')


Mounted at /content/drive
Python  : 3.12.13
PyTorch : 2.10.0+cu128
CUDA    : 12.8
GPU     : Tesla T4
VRAM    : 15.6 GB
RAM     : 13.6 GB total  |  11.7 GB free


In [7]:
import os, torch, psutil

# ================================================================
# Colab paths — all pointing to Google Drive
# ================================================================
BASE_DATA_DIR = '/content/drive/MyDrive/disease_data'
OUTPUT_DIR    = '/content/drive/MyDrive/medical_ai_models'
CACHE_PATH    = '/content/drive/MyDrive/disease_data/organ_cache.h5'
# ================================================================

IMG_SIZE      = 128
MAX_PER_CLASS = 2000
BATCH_SIZE    = 64
EPOCHS_P1     = 12
EPOCHS_P2     = 8
LR_P1         = 1e-4
LR_P2         = 3e-5
NUM_WORKERS   = 2      # Colab supports 2 workers safely

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMG_EXTS = ('.jpg','.jpeg','.png','.bmp','.tif','.tiff')

def count_images(root):
    n = 0
    for dirpath, _, files in os.walk(root):
        n += sum(1 for f in files if f.lower().endswith(IMG_EXTS))
    return n

def show_tree(root, max_depth=3, prefix=''):
    if not os.path.exists(root): return
    try: entries = sorted(os.listdir(root))
    except: return
    dirs = [e for e in entries if os.path.isdir(os.path.join(root, e))]
    for d in dirs[:6]:
        sub = os.path.join(root, d)
        n   = count_images(sub)
        print(f'{prefix}  {d}/  ({n:,} images)')
        if max_depth > 1 and n == 0:
            show_tree(sub, max_depth-1, prefix+'  ')
    if len(dirs) > 6:
        print(f'{prefix}  ... and {len(dirs)-6} more')

print('Dataset folder inspection:\n')
for folder in ['kvasir-dataset','bone_fracture','knee','dental','brain_mri','chest_xray','skin','eye']:
    path = os.path.join(BASE_DATA_DIR, folder)
    if not os.path.exists(path):
        print(f'  NOT FOUND: {folder}'); continue
    n = count_images(path)
    status = 'OK' if n > 0 else 'WARNING: 0 images'
    print(f'[{status}] {folder}/ — {n:,} images')
    if n == 0 or folder == 'dental':
        show_tree(path, max_depth=4)
    print()

free_gb = psutil.virtual_memory().available / 1e9
est_gb  = IMG_SIZE * IMG_SIZE * 3 * MAX_PER_CLASS * 25 / 1e9
print(f'Device   : {device}')
print(f'Cache est: ~{est_gb:.1f} GB  |  Free RAM: {free_gb:.1f} GB')
print(f'Strategy : {"FULL RAM LOAD" if free_gb > est_gb+2 else "HDF5 STREAMING"}')


Dataset folder inspection:

[OK] kvasir-dataset/ — 4,000 images

[OK] bone_fracture/ — 8,300 images

[OK] knee/ — 3,300 images

[OK] dental/ — 13,862 images
  Calculus/  (1,296 images)
  Caries_Gingivitus_ToothDiscoloration_Ulcer-yolo_annotated-Dataset/  (1,542 images)
  Data caries/  (2,601 images)
  Gingivitis/  (2,349 images)
  Mouth Ulcer/  (2,806 images)
  Tooth Discoloration/  (2,017 images)
  ... and 1 more

[OK] brain_mri/ — 7,200 images

[OK] chest_xray/ — 17,568 images

[OK] skin/ — 10,015 images

[OK] eye/ — 87,104 images

Device   : cuda
Cache est: ~2.5 GB  |  Free RAM: 9.4 GB
Strategy : FULL RAM LOAD


In [4]:
!pip install medmnist
import os, random, json, time
import medmnist
import numpy as np
import h5py, psutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.transforms import ToPILImage
from medmnist import OrganAMNIST, OrganCMNIST, OrganSMNIST
torch.manual_seed(42); np.random.seed(42); random.seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed(42)
print('Imports OK')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.7 MB/s eta 0:00:00
Imports OK


In [8]:
ORGAN_KEEP = {0:'Bladder',3:'Heart',4:'Kidney_Left',5:'Kidney_Right',
               6:'Liver',7:'Lung_Left',8:'Lung_Right',9:'Pancreas',10:'Spleen'}

medmnist_items = []
print('Loading OrganMNIST...')
for ds_cls, tag in [(OrganAMNIST,'axial'),(OrganCMNIST,'coronal'),(OrganSMNIST,'sagittal')]:
    for split in ['train','test']:
        ds = ds_cls(split=split, download=True, size=64)
        for idx in tqdm(range(len(ds)), desc=f'  {tag}/{split}', leave=False):
            img_arr, label_arr = ds[idx]
            label = int(label_arr[0]) if hasattr(label_arr,'__len__') else int(label_arr)
            if label not in ORGAN_KEEP: continue
            if isinstance(img_arr, Image.Image):
                pil = img_arr.convert('RGB')
            elif isinstance(img_arr, np.ndarray):
                pil = Image.fromarray(img_arr.astype(np.uint8)).convert('RGB')
            else:
                pil = transforms.ToPILImage()(img_arr).convert('RGB')
            medmnist_items.append((pil, ORGAN_KEEP[label]))
print(f'MedMNIST: {len(medmnist_items):,} items')

IMG_EXTS = ('.jpg','.jpeg','.png','.bmp','.tif','.tiff')
file_items = []
B = BASE_DATA_DIR

def collect(src, cls, max_n=None):
    if not os.path.exists(src): return 0
    files = [f for f in os.listdir(src) if f.lower().endswith(IMG_EXTS)]
    if max_n: files = files[:max_n]
    for f in files: file_items.append((os.path.join(src, f), cls))
    return len(files)

def collect_recursive(src, cls, max_n=None):
    if not os.path.exists(src): return 0
    all_files = []
    for dirpath, _, files in os.walk(src):
        for f in files:
            if f.lower().endswith(IMG_EXTS):
                all_files.append(os.path.join(dirpath, f))
    if max_n: all_files = all_files[:max_n]
    for path in all_files: file_items.append((path, cls))
    return len(all_files)

print('\n--- Collecting sources ---')

# ================================================================
# 1. KVASIR — GI tract
# kvasir-dataset/{folder}/
# ================================================================
for folder, organ in {
    'esophagitis':'Esophagus', 'normal-z-line':'Esophagus',
    'normal-pylorus':'Stomach',
    'polyps':'Colon', 'ulcerative-colitis':'Colon',
    'normal-cecum':'Colon', 'dyed-lifted-polyps':'Colon',
    'dyed-resection-margins':'Colon'}.items():
    collect(os.path.join(B,'kvasir-dataset',folder), organ)
print('  Kvasir done')

# ================================================================
# 2. BONE FRACTURE
# bone_fracture/BoneFractureYolo8/{train,valid}/images/
# bone_fracture/bone fracture detection.v4-v4.yolov8/{train,valid}/images/
# ================================================================
for yolo_dir in ['BoneFractureYolo8', 'bone fracture detection.v4-v4.yolov8']:
    for split in ['train','valid','test']:
        collect(os.path.join(B,'bone_fracture',yolo_dir,split,'images'), 'Bone', 1000)
print('  Bone done')

# ================================================================
# 3. KNEE X-RAY
# knee/MedicalExpert-I/{0Normal,1Doubtful,...}/
# knee/MedicalExpert-II/{0Normal,1Doubtful,...}/
# ================================================================
for expert in ['MedicalExpert-I', 'MedicalExpert-II']:
    for cls in ['0Normal','1Doubtful','2Mild','3Moderate','4Severe']:
        collect(os.path.join(B,'knee',expert,cls), 'Knee', 300)
print('  Knee done')

# ================================================================
# 4. DENTAL
# dental/Calculus/Calculus/
# dental/Gingivitis/Gingivitis/
# dental/hypodontia/hypodontia/
# dental/Mouth Ulcer/Mouth Ulcer/ulcer original dataset/ulcer original dataset/
# dental/Tooth Discoloration/Tooth Discoloration/tooth discoloration original dataset/tooth discoloration original dataset/
# dental/Data caries/Data caries/caries orignal data set/done/
# dental/Caries_Gingivitus.../Data/images/{train,val}/  (YOLO format)
# ================================================================
dental_sources = [
    os.path.join(B,'dental','Calculus','Calculus'),
    os.path.join(B,'dental','Gingivitis','Gingivitis'),
    os.path.join(B,'dental','hypodontia','hypodontia'),
    os.path.join(B,'dental','Mouth Ulcer','Mouth Ulcer','ulcer original dataset','ulcer original dataset'),
    os.path.join(B,'dental','Tooth Discoloration','Tooth Discoloration','tooth discoloration original dataset','tooth discoloration original dataset'),
    os.path.join(B,'dental','Data caries','Data caries','caries orignal data set','done'),
]
dental_yolo_base = os.path.join(B,'dental',
    'Caries_Gingivitus_ToothDiscoloration_Ulcer-yolo_annotated-Dataset',
    'Caries_Gingivitus_ToothDiscoloration_Ulcer-yolo_annotated-Dataset','Data')
for split in ['train','val']:
    dental_sources.append(os.path.join(dental_yolo_base,'images',split))

dental_found = sum(collect(src, 'Teeth', 500) for src in dental_sources)
if dental_found == 0:
    print('  Dental: known paths failed — doing recursive scan')
    dental_found = collect_recursive(os.path.join(B,'dental'), 'Teeth', 3000)
print(f'  Dental: {dental_found:,} images')

# ================================================================
# 5. BRAIN MRI
# brain_mri/Training/{glioma,meningioma,notumor,pituitary}/
# brain_mri/Testing/{glioma,meningioma,notumor,pituitary}/
# All tumor types → class 'Brain' (organ location classifier)
# ================================================================
brain_found = 0
for split in ['Training','Testing']:
    for cls in ['glioma','meningioma','notumor','pituitary']:
        brain_found += collect(os.path.join(B,'brain_mri',split,cls), 'Brain', 500)
print(f'  Brain: {brain_found:,} images')

# ================================================================
# 6. CHEST X-RAY  → Lung_Chest class
# chest_xray/train/{NORMAL,PNEUMONIA}/
# chest_xray/test/{NORMAL,PNEUMONIA}/
# chest_xray/val/{NORMAL,PNEUMONIA}/
# Using top-level splits (not the nested chest_xray/chest_xray/ duplicate)
# ================================================================
chest_found = 0
for split in ['train','test','val']:
    for cls in ['NORMAL','PNEUMONIA']:
        chest_found += collect(os.path.join(B,'chest_xray',split,cls), 'Lung_Chest', 800)
print(f'  Chest X-ray (Lung_Chest): {chest_found:,} images')

# ================================================================
# 7. SKIN — HAM10000
# skin/HAM10000_images_part_1/
# skin/HAM10000_images_part_2/
# ================================================================
skin_found = 0
for part in ['HAM10000_images_part_1','HAM10000_images_part_2']:
    skin_found += collect(os.path.join(B,'skin',part), 'Skin', 1500)
print(f'  Skin (HAM10000): {skin_found:,} images')

# ================================================================
# 8. EYE / RETINA
# eye/{condition_folder}/  — direct top-level folders
# eye/ROFT/Retinal OCT/{AMD,DME,...}/
# eye/ROFT/Cornea_Ulcers/ + Corneal_Ulcers_Agumentation/
# ================================================================
for eye_cls, folders in {
    'Eye_Normal'      : ['0.0.Normal'],
    'Eye_DR'          : ['0.3.DR1','1.0.DR2','1.1.DR3',
                          '29.0.Blur fundus without PDR','29.1.Blur fundus with suspected PDR'],
    'Eye_Glaucoma'    : ['10.0.Possible glaucoma','0.2.Large optic cup','10.1.Optic atrophy'],
    'Eye_Maculopathy' : ['6.Maculopathy','20.Massive hard exudates','5.0.CSCR','5.1.VKH disease',
                          '21.Yellow-white spots-flecks'],
    'Eye_Vascular'    : ['2.0.BRVO','2.1.CRVO','3.RAO','23.Vessel tortuosity','25.Preretinal hemorrhage'],
    'Eye_Degeneration': ['15.0.Retinitis pigmentosa','9.Pathological myopia','4.Rhegmatogenous RD',
                          '15.1.Bietti crystalline dystrophy','16.Peripheral retinal degeneration and break'],
    'Eye_Other'       : ['7.ERM','8.MH','11.Severe hypertensive retinopathy',
                          '12.Disc swelling and elevation','13.Dragged Disc',
                          '14.Congenital disc abnormality','17.Myelinated nerve fiber',
                          '18.Vitreous particles','19.Fundus neoplasm','22.Cotton-wool spots',
                          '24.Chorioretinal atrophy-coloboma','26.Fibrosis',
                          '27.Laser Spots','28.Silicon oil in eye','0.1.Tessellated fundus'],
}.items():
    for folder in folders:
        collect(os.path.join(B,'eye',folder), eye_cls, 200)

for f in ['AMD','DME','ERM','NO','RAO','RVO','VID']:
    collect(os.path.join(B,'eye','ROFT','Retinal OCT',f), 'Eye_OCT', 150)
for f in ['Cornea_Ulcers','Corneal_Ulcers_Agumentation']:
    collect(os.path.join(B,'eye','ROFT',f), 'Eye_Cornea', 300)
print(f'  Eye done')

# ================================================================
# SUMMARY
# ================================================================
all_classes = sorted(set([c for _,c in file_items]+[c for _,c in medmnist_items]))
all_raw     = Counter([c for _,c in file_items]+[c for _,c in medmnist_items])
print(f'\nTotal classes : {len(all_classes)}')
print(f'Total images  : {len(file_items)+len(medmnist_items):,}')
print('\nPer-class counts (before cap):')
for cls in all_classes:
    n = all_raw[cls]
    warn = '  <-- WARNING: 0 images!' if n == 0 else ''
    print(f'  {cls:25s}: {n:,}{warn}')


Loading OrganMNIST...


MedMNIST: 87,771 items

--- Collecting sources ---
  Kvasir done
  Bone done
  Knee done
  Dental: 2,716 images
  Brain: 3,600 images
  Chest X-ray (Lung_Chest): 2,240 images
  Skin (HAM10000): 3,000 images
  Eye done

Total classes : 27
Total images  : 111,275

Per-class counts (before cap):
  Bladder                  : 6,927
  Bone                     : 3,034
  Brain                    : 3,600
  Colon                    : 2,500
  Esophagus                : 1,000
  Eye_Cornea               : 600
  Eye_DR                   : 265
  Eye_Degeneration         : 155
  Eye_Glaucoma             : 75
  Eye_Maculopathy          : 144
  Eye_Normal               : 38
  Eye_OCT                  : 796
  Eye_Other                : 217
  Eye_Vascular             : 106
  Heart                    : 4,511
  Kidney_Left              : 9,678
  Kidney_Right             : 9,499
  Knee                     : 2,518
  Liver                    : 19,812
  Lung_Chest               : 2,240
  Lung_Left              

In [ ]:
# ================================================================
# BUILD HDF5 CACHE  - run ONCE (~15-30 min)
# Reads images from disk once, resizes, saves compressed.
# Cache: 128px ~ 1.5 GB  |  224px ~ 4.5 GB
# ================================================================
if os.path.exists(CACHE_PATH):
    print(f'Cache already exists: {CACHE_PATH}')
    print(f'Size: {os.path.getsize(CACHE_PATH)/1e9:.2f} GB - skipping rebuild.')
    print('Delete the file to force rebuild.')
else:
    print(f'Building cache -> {CACHE_PATH}')
    class_to_idx  = {c: i for i, c in enumerate(all_classes)}
    class_buckets = {cls: [] for cls in all_classes}
    for path, cls in file_items: class_buckets[cls].append(('file', path))
    for pil,  cls in medmnist_items: class_buckets[cls].append(('pil', pil))

    capped = []
    for cls in all_classes:
        bucket = class_buckets[cls]; random.shuffle(bucket)
        for item in bucket[:MAX_PER_CLASS]: capped.append((item, class_to_idx[cls]))
    random.shuffle(capped)
    N = len(capped)
    print(f'Total after cap: {N:,}')
    cc = Counter([lbl for _,lbl in capped])
    for i,cls in enumerate(all_classes): print(f'  {cls:25s}: {cc[i]:,}')

    with h5py.File(CACHE_PATH, 'w') as hf:
        imgs_ds = hf.create_dataset('images', shape=(N,IMG_SIZE,IMG_SIZE,3),
                                    dtype='uint8', chunks=(64,IMG_SIZE,IMG_SIZE,3), compression='lzf')
        lbls_ds = hf.create_dataset('labels', shape=(N,), dtype='int32')
        hf.attrs['classes']     = json.dumps(all_classes)
        hf.attrs['img_size']    = IMG_SIZE
        hf.attrs['max_per_cls'] = MAX_PER_CLASS
        errors = 0
        t0 = time.time()
        for i, ((src_type, src_data), label) in enumerate(tqdm(capped, desc='Caching', ncols=80)):
            try:
                img = Image.open(src_data).convert('RGB') if src_type == 'file' else src_data.convert('RGB')
                imgs_ds[i] = np.array(img.resize((IMG_SIZE,IMG_SIZE), Image.BILINEAR), dtype=np.uint8)
                lbls_ds[i] = label
            except Exception:
                imgs_ds[i] = np.zeros((IMG_SIZE,IMG_SIZE,3), dtype=np.uint8); lbls_ds[i] = label; errors += 1
    print(f'Done in {(time.time()-t0)/60:.1f} min  |  {os.path.getsize(CACHE_PATH)/1e9:.2f} GB  |  {errors} errors')

Building cache -> /content/drive/MyDrive/disease_data/organ_cache.h5
Total after cap: 35,896
  Bladder                  : 2,000
  Bone                     : 2,000
  Brain                    : 2,000
  Colon                    : 2,000
  Esophagus                : 1,000
  Eye_Cornea               : 600
  Eye_DR                   : 265
  Eye_Degeneration         : 155
  Eye_Glaucoma             : 75
  Eye_Maculopathy          : 144
  Eye_Normal               : 38
  Eye_OCT                  : 796
  Eye_Other                : 217
  Eye_Vascular             : 106
  Heart                    : 2,000
  Kidney_Left              : 2,000
  Kidney_Right             : 2,000
  Knee                     : 2,000
  Liver                    : 2,000
  Lung_Chest               : 2,000
  Lung_Left                : 2,000
  Lung_Right               : 2,000
  Pancreas                 : 2,000
  Skin                     : 2,000
  Spleen                   : 2,000
  Stomach                  : 500
  Teeth            

Caching:  35%|████████▌               | 12741/35896 [1:13:52<2:16:32,  2.83it/s]

In [ ]:
with h5py.File(CACHE_PATH, 'r') as hf:
    all_classes   = json.loads(hf.attrs['classes'])
    stored_N      = hf['images'].shape[0]
    cache_size_gb = os.path.getsize(CACHE_PATH) / 1e9

class_to_idx = {c: i for i, c in enumerate(all_classes)}
free_ram_gb  = psutil.virtual_memory().available / 1e9
USE_RAM      = free_ram_gb > cache_size_gb + 2.0
print(f'Cache: {stored_N:,} samples, {cache_size_gb:.2f} GB  |  Free RAM: {free_ram_gb:.1f} GB')

if USE_RAM:
    print('Strategy: FULL RAM LOAD (fastest - zero disk I/O per epoch)')
    with h5py.File(CACHE_PATH, 'r') as hf:
        images_np = hf['images'][:]
        labels_np = hf['labels'][:]
    print(f'Loaded into RAM: {images_np.nbytes/1e9:.2f} GB')
else:
    print('Strategy: HDF5 STREAMING (SSD reads per batch - still fast)')
    images_np = labels_np = None

with h5py.File(CACHE_PATH, 'r') as hf: lbl_check = hf['labels'][:]
counts = Counter(lbl_check.tolist())
print('Per-class counts:')
for i,cls in enumerate(all_classes): print(f'  {cls:25s}: {counts[i]:,}')

In [ ]:
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08)),
])
val_aug = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

class CachedOrganDataset(Dataset):
    def __init__(self, indices, images_np, labels_np, h5_path, transform=None):
        self.indices = indices; self.images_np = images_np
        self.labels_np = labels_np; self.h5_path = h5_path
        self.transform = transform; self._hf = None
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        if self.images_np is not None:
            img_arr = self.images_np[real_idx]; label = int(self.labels_np[real_idx])
        else:
            if self._hf is None: self._hf = h5py.File(self.h5_path, 'r')
            img_arr = self._hf['images'][real_idx]; label = int(self._hf['labels'][real_idx])
        img = Image.fromarray(img_arr)
        return self.transform(img) if self.transform else img, label

with h5py.File(CACHE_PATH, 'r') as hf: all_labels_for_split = hf['labels'][:]
all_indices = np.arange(len(all_labels_for_split))
train_idx, val_idx = train_test_split(all_indices, test_size=0.15,
                                       stratify=all_labels_for_split, random_state=42)
_lbl = labels_np if USE_RAM else all_labels_for_split
train_ds = CachedOrganDataset(train_idx, images_np, _lbl, CACHE_PATH, train_aug)
val_ds   = CachedOrganDataset(val_idx,   images_np, _lbl, CACHE_PATH, val_aug)

# num_workers=0 + persistent_workers=False required on Windows
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))
print(f'Classes: {len(all_classes)}  Train: {len(train_ds):,}  Val: {len(val_ds):,}')

In [ ]:
# Expected: RAM mode >500 bps | SSD streaming >50 bps
print('Benchmarking DataLoader speed...')
it = iter(train_loader); _ = next(it)
t0 = time.time()
for _ in range(20): imgs, labels = next(it)
bps = 20 / (time.time() - t0)
est_min = len(train_loader) / bps / 60
print(f'  {bps:.1f} batches/sec  |  ~{est_min:.1f} min/epoch (data only)  |  ~{est_min*1.4:.1f} min with GPU')
if bps < 20: print('Still slow - reduce MAX_PER_CLASS to 500 in Cell 2 and rebuild cache')
else: print('Good speed - proceed to training')
del it

In [ ]:
# ================================================================
# CELL 9b — OPTIONAL: Copy cache from Drive → Colab local disk
# Run this if Cell 8 benchmark shows < 10 batches/sec
# Colab local SSD is 10-20x faster than Drive
# Takes 2-5 min but pays off across all epochs
# ================================================================
import shutil
LOCAL_CACHE = '/content/organ_cache_local.h5'

if not os.path.exists(LOCAL_CACHE):
    print(f'Copying cache from Drive to Colab local disk...')
    print(f'  {CACHE_PATH} -> {LOCAL_CACHE}')
    shutil.copy2(CACHE_PATH, LOCAL_CACHE)
    print(f'  Done: {os.path.getsize(LOCAL_CACHE)/1e9:.2f} GB')
else:
    print(f'Local cache already exists: {os.path.getsize(LOCAL_CACHE)/1e9:.2f} GB')

# Point datasets to local copy
CACHE_PATH = LOCAL_CACHE
train_ds.h5_path = LOCAL_CACHE
val_ds.h5_path   = LOCAL_CACHE
print('Datasets now using local cache')
print('Re-run Cell 8 to verify speed improvement')


In [ ]:
def build_model(num_classes, freeze=True):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    if freeze:
        for name, p in m.named_parameters():
            if 'layer4' not in name and 'fc' not in name: p.requires_grad = False
    in_f = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_f, 512),
                          nn.ReLU(), nn.BatchNorm1d(512),
                          nn.Dropout(0.3), nn.Linear(512, num_classes))
    return m

model = build_model(len(all_classes), freeze=True).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'ResNet50 | Classes: {len(all_classes)} | Trainable: {trainable:,} / {total:,}')

In [ ]:
class_counts  = [counts[i] for i in range(len(all_classes))]
class_weights = torch.tensor(
    [sum(class_counts)/(len(all_classes)*max(c,1)) for c in class_counts],
    dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                         lr=LR_P1, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_P1, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
BEST_PATH = os.path.join(OUTPUT_DIR, 'organ_v2_best.pth')
print('Loss/Optimizer/Scaler ready')
for n,w in zip(all_classes, class_weights.cpu()): print(f'  {n:25s}: {w:.3f}')

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  Train', leave=False, ncols=80):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            out = model(imgs); loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item(); correct += (out.argmax(1)==labels).sum().item(); total += labels.size(0)
    return loss_sum/len(loader), correct/total

def val_epoch(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='  Val  ', leave=False, ncols=80):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                out = model(imgs); loss_sum += criterion(out, labels).item()
            p = out.argmax(1)
            correct += (p==labels).sum().item(); total += labels.size(0)
            preds_all.extend(p.cpu().numpy()); labels_all.extend(labels.cpu().numpy())
    return loss_sum/len(loader), correct/total, balanced_accuracy_score(labels_all, preds_all)

print('Train/val ready (torch.amp - no FutureWarning)')

In [ ]:
history  = {k: [] for k in ['tl','vl','ta','va','ba']}
best_acc = 0.0
print(f'PHASE 1 - Frozen backbone | {EPOCHS_P1} epochs | LR {LR_P1}')

for epoch in range(1, EPOCHS_P1+1):
    t0 = time.time()
    tl,ta        = train_epoch(model, train_loader, criterion, optimizer, scaler)
    vl,va,bal_a  = val_epoch(model, val_loader, criterion)
    scheduler.step()
    for k,v in zip(['tl','vl','ta','va','ba'],[tl,vl,ta,va,bal_a]): history[k].append(v)
    saved = ''
    if va > best_acc: best_acc=va; torch.save(model.state_dict(), BEST_PATH); saved='  saved'
    print(f'Ep {epoch:02d}/{EPOCHS_P1} | Train {ta*100:.1f}% {tl:.4f} | '
          f'Val {va*100:.1f}% bal {bal_a*100:.1f}% {vl:.4f} | '
          f'LR {optimizer.param_groups[0]["lr"]:.1e} | {(time.time()-t0)/60:.1f}min{saved}')

print(f'Phase 1 done - best val: {best_acc*100:.2f}%')

In [ ]:
model.load_state_dict(torch.load(BEST_PATH, map_location=device, weights_only=True))
for p in model.parameters(): p.requires_grad = True
backbone_p   = [p for n,p in model.named_parameters() if 'fc' not in n]
classifier_p = [p for n,p in model.named_parameters() if 'fc' in n]
opt2 = optim.AdamW([{'params':backbone_p,'lr':LR_P2 * 0.2},{'params':classifier_p,'lr':LR_P2}], weight_decay=1e-4)
sch2    = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS_P2, eta_min=1e-8)
scaler2 = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
print(f'PHASE 2 - Full fine-tune | {EPOCHS_P2} epochs | Backbone LR {LR_P2 * 0.2:.1e} | Head LR {LR_P2:.1e}')

for epoch in range(1, EPOCHS_P2+1):
    t0 = time.time()
    tl,ta        = train_epoch(model, train_loader, criterion, opt2, scaler2)
    vl,va,bal_a  = val_epoch(model, val_loader, criterion)
    sch2.step()
    for k,v in zip(['tl','vl','ta','va','ba'],[tl,vl,ta,va,bal_a]): history[k].append(v)
    saved = ''
    if va > best_acc: best_acc=va; torch.save(model.state_dict(), BEST_PATH); saved='  saved'
    print(f'Ep {epoch:02d}/{EPOCHS_P2} | Train {ta*100:.1f}% {tl:.4f} | '
          f'Val {va*100:.1f}% bal {bal_a*100:.1f}% {vl:.4f} | '
          f'LR {opt2.param_groups[1]["lr"]:.1e} | {(time.time()-t0)/60:.1f}min{saved}')

print(f'Phase 2 done - best val: {best_acc*100:.2f}%')

In [ ]:
# Training curves
ep = range(1, EPOCHS_P1+EPOCHS_P2+1)
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Organ Classifier v2', fontsize=13, fontweight='bold')
axes[0].plot(ep,history['tl'],'#534AB7',label='Train',lw=2)
axes[0].plot(ep,history['vl'],'#D85A30',label='Val',lw=2)
axes[0].axvline(EPOCHS_P1+0.5,color='gray',ls='--',alpha=0.5); axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep,[a*100 for a in history['ta']],'#534AB7',label='Train',lw=2)
axes[1].plot(ep,[a*100 for a in history['va']],'#D85A30',label='Val',lw=2)
axes[1].axvline(EPOCHS_P1+0.5,color='gray',ls='--',alpha=0.5); axes[1].set_title('Raw Acc %'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(ep,[a*100 for a in history['ba']],'#1D9E75',label='Balanced',lw=2)
axes[2].axhline(best_acc*100,color='orange',ls=':',label=f'Best {best_acc*100:.1f}%')
axes[2].axvline(EPOCHS_P1+0.5,color='gray',ls='--',alpha=0.5); axes[2].set_title('Balanced Acc %'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'organ_v2_curves.png'),dpi=120); plt.close()
print('Curves saved')

# Final eval
model.load_state_dict(torch.load(BEST_PATH, map_location=device, weights_only=True)); model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc='Eval', ncols=80):
        out = model(imgs.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy()); all_labels.extend(labels.numpy())
all_preds = np.array(all_preds); all_labels = np.array(all_labels)
bal = balanced_accuracy_score(all_labels, all_preds)
print(classification_report(all_labels, all_preds, target_names=all_classes, digits=3))

cm_norm = confusion_matrix(all_labels,all_preds).astype(float)
cm_norm /= cm_norm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(16,13))
sns.heatmap(cm_norm,annot=True,fmt='.2f',cmap='Blues',xticklabels=all_classes,yticklabels=all_classes,ax=ax)
ax.set_title(f'Confusion Matrix - Balanced Acc: {bal*100:.2f}%', fontsize=13, fontweight='bold')
plt.xticks(rotation=35,ha='right'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'organ_v2_cm.png'),dpi=120); plt.close()

# Save
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR,'organ_model_v2.pth'))
meta = {'model_name':'ResNet50 Organ Classifier v2','version':'2.0',
        'num_classes':len(all_classes),'classes':all_classes,'class_to_idx':class_to_idx,
        'input_size':[IMG_SIZE,IMG_SIZE],'normalize_mean':_MEAN,'normalize_std':_STD,
        'best_val_accuracy':round(best_acc*100,2),'balanced_accuracy':round(bal*100,2)}
with open(os.path.join(OUTPUT_DIR,'organ_metadata_v2.json'),'w') as f: json.dump(meta,f,indent=2)
print(f'Saved to {OUTPUT_DIR}')
print(f'Best val: {best_acc*100:.2f}%  |  Balanced: {bal*100:.2f}%')

In [ ]:
def load_organ_model(weights_path, metadata_path, device=None):
    if device is None: device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    with open(metadata_path) as f: meta = json.load(f)
    m = models.resnet50(weights=None)
    in_f = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_f,512),
                          nn.ReLU(), nn.BatchNorm1d(512),
                          nn.Dropout(0.3), nn.Linear(512,meta['num_classes']))
    m.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    return m.eval().to(device), meta

def predict_organ(model, image_path, meta, device=None):
    if device is None: device = next(model.parameters()).device
    size = meta['input_size'][0]
    tf = transforms.Compose([transforms.Resize((size,size)), transforms.ToTensor(),
                               transforms.Normalize(meta['normalize_mean'], meta['normalize_std'])])
    with torch.no_grad():
        probs = torch.softmax(model(tf(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)),1)[0].cpu().numpy()
    idx = probs.argmax()
    return {'organ':meta['classes'][idx],'confidence':float(probs[idx]),
            'all_probs':{c:float(p) for c,p in zip(meta['classes'],probs)}}

print('Inference helper ready')
print('Usage:')
print('  model, meta = load_organ_model(weights_path, metadata_path)')
print('  result = predict_organ(model, image_path, meta)')
print('  print(result["organ"], result["confidence"])')